# Project OverView

The project was developed for Enbridge, a leading Canadian energy infrastructure company with a significant presence in the renewable energy sector. The objective was to build an AI-powered assistant for wind turbine field engineers responsible for inspecting, maintaining, and repairing wind turbines.

The assistant enabled engineers to quickly access critical information such as maintenance manuals, troubleshooting guides, standard operating procedures (SOPs), safety regulations, and historical maintenance records through natural language queries. By leveraging Generative AI and Retrieval-Augmented Generation (RAG), the solution delivered accurate, context-aware responses from the organization's knowledge base, reducing the need to manually search through large volumes of technical documentation.

This improved field productivity, reduced downtime, accelerated issue resolution, and helped engineers make informed maintenance decisions while ensuring compliance with safety standards.



# RAG

## Indexing Stage

### Azure Blog Storage

All company documents (manuals, reports, diagrams, and safety guidelines) are stored in Microsoft Azure Blob Storage, which provides centralized, scalable, and secure storage

### Azure Function

Whenever a new document is uploaded to Azure Blob Storage, an Azure Function is automatically triggered. This function checks the document and adds a task to an Azure Storage Queue. Another Azure Function reads the task from the queue, opens the document, extracts the content, and sends it to the AI pipeline.

### Azure Document Intelligence

Azure Document Intelligence extracts content from PDFs, scanned files, and images. It identifies document structure (headings, paragraphs, tables, lists) and uses OCR(Optical Character Recognition )  to extract text from scanned documents and diagrams.

The output is converted into structured Markdown instead of plain text, preserving headers and document hierarchy for better chunking.

### PyMuPDF 

If the PDF contains images or diagrams, we use PyMuPDF to extract them and save them separately in Azure Blob Storage. We also store metadata, such as the image name and page number, so each image remains linked to the relevant text.

When a user asks a question, the RAG system retrieves both the relevant text and the associated image references. FastAPI then generates a secure SAS URL so the images can be displayed along with the answer, providing a multimodal response.

### Chucking Strategy 

First, the extracted Markdown content is split based on document headings using LangChain's MarkdownHeaderTextSplitter. This preserves the document's logical structure, ensuring that related sections such as maintenance procedures, safety instructions, and alarm descriptions remain together.

Next, the chunks are optimized using token-based chunking. Smaller sections are merged until they reach the desired chunk size, while larger sections are further divided using RecursiveCharacterTextSplitter to stay within the LLM's token limits.

To preserve context, the system applies chunk overlap, where a small portion of the previous chunk is repeated in the next chunk. This helps maintain continuity when information spans multiple chunks.

After the chunks are created, Azure OpenAI extracts structured metadata for each chunk, such as the procedure name, procedure code, alarm code, related figures or tables, and a short summary.

Finally, each chunk, along with its metadata, is stored in a structured JSON format. 

### Azure OpenAI Embedding

In this step, each document chunk is converted into a vector embedding using text-embedding-ada-002. The embedding is a 1536-dimensional vector that captures the semantic meaning of the text.

This allows the system to perform semantic search, where similar meanings can be matched even if the words are different. All document chunks are converted into embeddings and stored.

### Azure AI Search Index

After generating embeddings, we store each document chunk in Azure AI Search along with its metadata and embedding vector. Azure AI Search indexes both the text and the vector, enabling keyword search as well as semantic search. This helps retrieve the most relevant document chunks for the user's query.

## Retrieval Stage

### authentication and authorization 
Every request first goes through authentication to verify the user's identity and authorization to check their permissions.

### Azure cache Redis Rate Limit

We use Azure Azure cache Redis for rate limiting.It+ checks the number of requests a user sends (for example, 10 requests per minute). If the limit is exceeded, it returns an HTTP 429 (Too Many Requests) response without forwarding the request to FastAPI or Azure OpenAI, protecting the application from abuse and reducing unnecessary LLM costs.

### Azure cache Redis (token limit )

We use Azure Cache for Redis to track each user's daily token usage and block requests that exceed the configured limit before they reach the LLM, helping control costs and prevent misuse.

### Azure AI Language 

Azure AI Language is mainly used for PII detection. It identifies sensitive information such as engineer names, email IDs, employee IDs, and other personal data in user queries or documents. We redact this information before sending the request to the LLM, while preserving technical information like turbine IDs, alarm codes, and component names because they are required for accurate retrieval.

### Azure AI Content Safety guardrails

We apply Azure AI Content Safety guardrails at both input and output stages. Before retrieval, we validate and filter user queries to prevent harmful inputs and prompt injection attacks. After Azure OpenAI generates the response, we apply output guardrails to detect sensitive information, harmful content, or policy violations before sending the answer back to the user.


### Pydantic and Regex Validation 

We first use Pydantic to validate the request schema. It checks that all required fields are present. After that, we use Regex validation to verify specific input patterns, such as conversation IDs or allowed characters, and to sanitize the input if necessary. Together, these validations ensure that only valid and well-formed requests enter the RAG pipeline.

### Query rewrite 

we determine whether the query is a new question or a follow-up. If it's a follow-up, we retrieve the relevant conversation history from Azure Cosmos DB. We then combine the current query with the conversation context and use an LLM to rewrite it into a clear, standalone question. The rewritten query is converted into an embedding and sent to Azure AI Search

### Azure AI Search 

In this stage, the system performs hybrid retrieval using Azure AI Search to find the most relevant document chunks.

First, keyword search is performed using BM25, which retrieves chunks containing exact matches of the query terms. This is especially useful for technical queries involving error codes, component IDs, or specific terminology.

At the same time, the query is converted into an embedding, and vector search is performed using KNN (K-Nearest Neighbors). The system retrieves the Top-K nearest document vectors based on cosine similarity, allowing it to find semantically similar content even when different words are used.

The results from both keyword and vector search are then combined using RRF (Reciprocal Rank Fusion), which produces a single ranked list of relevant document chunks.

Next, Semantic Ranker is applied to re-score these retrieved chunks based on their relevance to the user's query. This improves retrieval accuracy by promoting the most relevant chunks to the top.

The final Top-K high-quality chunks are then passed to Azure OpenAI to generate an accurate response.


## Generative Stage

### LLM response 

After retrieval, the final Top-K relevant document chunks are combined with the user's query, conversation history, and a carefully designed system prompt. This complete prompt is sent to Azure OpenAI (GPT-4o). The LLM generates a grounded response using only the retrieved context instead of relying on its own knowledge. The system prompt instructs the model to answer only from the provided documents, avoid hallucinations, and clearly state when sufficient information is not available.

### Azure AI content Safety

the FastAPI backend performs post-processing before returning it to the user. First, Azure AI Content Safety validates the generated output to ensure it does not contain harmful or policy-violating content. 

### Pydantic scheme validation and regrex

we first validate the structured JSON using a Pydantic schema to ensure all required fields and data types are correct. We then apply regex-based sanitization to clean any unwanted formatting or extra text. Finally, returning the playload response to the frontend.


# Security

## Auth 

Authentication verifies the identity of every user calling the API using OAuth 2.0 and JSON Web Tokens (JWT) issued by Microsoft Azure Entra ID (formerly Azure Active Directory).

How Authentication works step-by-step:

Step 1: Client Login and Token Acquisition
The user logs into the web application using their corporate Microsoft account via Azure Entra ID. Upon successful login, Azure Entra ID issues a cryptographically signed Bearer JWT token to the user's browser.

Step 2: Request Authorization Header
The client web application attaches the Bearer token to the Authorization header of every incoming API request.

Step 3: Cryptographic Token Verification in FastAPI
When the HTTP request reaches FastAPI:
1. The validate_user dependency extracts the Bearer token.
2. FastAPI verifies the token signature against Microsoft Entra ID public OpenID keys.
3. It verifies that the token is not expired and that the audience claim matches your registered Application Client ID.

Step 4: Claims Extraction
FastAPI extracts key user identity details directly from the validated token:
- User Object ID (Azure OID): The permanent unique identifier for the user.
- User Email: Corporate email address.
- User Display Name: Full name of the user.
- User Role: Assigned enterprise role (such as Engineer, Tester, or Admin).

Step 5: Database Synchronization
FastAPI queries Cosmos DB using the User Object ID:
- If the user has logged in before, FastAPI retrieves their existing user profile.
- If the user is logging in for the first time, FastAPI automatically provisions a new user profile document in Cosmos DB.
- For local offline development, a fallback developer user ID is provided so testing works without a live identity provider.

Step 6: Secure Execution
Once authenticated, the User object containing the user ID and assigned role is passed down to route handlers, sliding-window rate limiters, and daily token budget checkers. Requests with missing or invalid tokens are immediately blocked with an HTTP 401 Unauthorized error.

## Authentication

Our application uses Microsoft Entra ID in a single-tenant configuration, so only employees from our organization can access the AI Assistant. During authentication, the React frontend redirects unauthenticated users to Microsoft Entra ID, where they sign in using their corporate credentials and complete MFA if required. After successful authentication, Microsoft Entra ID issues an ID token for the frontend and an access token (JWT) for the backend. Every API request includes the JWT, and the FastAPI backend validates its signature, issuer, audience, tenant ID, and expiration time.

## Authorization
After authentication, the backend performs authorization using Role-Based Access Control (RBAC). It extracts the user's role or Azure AD group from the JWT and checks whether the user has permission to perform the requested operation. For example, engineers can query the AI Assistant, while administrators can also upload documents and manage the search index. If the user is authorized, the request enters the AI pipeline. If authorization fails, the backend immediately returns a 403 Forbidden response.

## Azure AI Language 

PII (Personally Identifiable Information) Detection identifies personally identifiable information such as names, email addresses, phone numbers, passport numbers, and credit card numbers. It returns the detected entities along with their categories and confidence scores. PII Redaction builds on detection by masking or replacing the sensitive information before it is stored, logged, or sent to an LLM. Detection is used for analysis, while redaction is used for privacy protection.

## Azure AI Content Safety 

is a service that detects, filters, and blocks harmful, unsafe, or policy-violating content in AI applications. It helps ensure that both user inputs and LLM-generated outputs comply with safety policies. It also helps detect prompt injection and jailbreak attempts.


Prompt Injection is an attack where malicious instructions are inserted into user prompts or external content, such as RAG documents, to manipulate the application's behavior. 
Jailbreak is an attempt to bypass the LLM's built-in safety policies and generate responses that would normally be restricted.

# Observability Metrics

We use **Azure Monitor + Application Insights** to continuously monitor the health, performance, reliability, and cost of our RAG chatbot. These metrics provide aggregated insights across all requests and help us create dashboards, alerts, and capacity planning.

Here are the exact metrics you will monitor directly inside the Microsoft Azure Portal across your deployed Azure services:

1. Azure OpenAI Service Metrics (Azure Monitor)
- Total Tokens Consumed: Total count of prompt tokens and completion tokens used per deployment (gpt-4o and text-embedding-3-small).
- OpenAI Throttling Errors (HTTP 429): Number of requests throttled because your application exceeded Azure OpenAI Tokens Per Minute (TPM) or Requests Per Minute (RPM) limits.
- Model Latency: Time to First Token (TTFT) and total response generation duration measured in milliseconds.

2. Azure Cache for Redis Enterprise Metrics (Azure Redis Metrics)
- Cache Hit Ratio: Percentage of requests resulting in Cache Hits versus Cache Misses.
- Memory Usage: Percentage of allocated Redis RAM used by stored vectors, chat history, and key indexes.
- RediSearch Vector Latency: Execution time in milliseconds for FT.SEARCH HNSW vector queries.
- CPU Utilization: Host CPU usage percentage for Redis Enterprise cluster nodes.

3. Azure Cosmos DB NoSQL Metrics (Cosmos DB Metrics)
- Request Units (RU/s) Consumption: Total RUs consumed per second for reading user profiles and writing chat history messages.
- RU Throttling (HTTP 429): Number of database operations blocked due to exceeding provisioned RU/s throughput.
- Cosmos DB Latency: Read and write latency in milliseconds (target is under 10 ms).
- Storage Growth: Total storage size used by documents in Megabytes or Gigabytes.

4. Azure AI Content Safety Metrics (Azure AI Services)
- Blocked Request Volume: Count of requests blocked due to safety violations (severity score 4 or higher).
- Category Breakdown: Distribution of flagged content across Hate, SelfHarm, Sexual, and Violence categories.

5. Azure AI Language PII Detection Metrics (Cognitive Services Metrics)
- Text Analytics API Calls: Total number of API calls and document character volumes processed.
- Detected PII Types: Count of PII entities detected and redacted per category (Email, Phone, Credit Card, SSN).

6. Azure App Service / Azure Container Apps Metrics (Host Metrics)
- CPU and Memory Utilization: Host CPU percentage and RAM usage for your FastAPI backend container.
- HTTP Status Code Distribution: Ratio of 200 OK responses versus 429 Rate Limit Throttled responses and 500 Server Errors.
- Application Insights Live Metrics Stream: Real-time telemetry, dependency call duration, and OpenTelemetry trace logs forwarded via your Application Insights connection string.




Here are the key important metrics you will be monitoring for your Enterprise Multi-Agent AI Assistant project, grouped into 5 core operational areas:

Multi-Agent Engine and Tool Orchestration Metrics
Dual LLM Provider Ratio: Tracks usage split between Azure OpenAI and your local Ollama engine (gemma4:31b-cloud).
Agent Tool Call Success and Error Rates: Invocation volume, response speed, and error rates for the Calculator tool, Tavily Web Search tool, and Filesystem MCP Server.
Multi-Turn Execution Steps: Number of reasoning steps taken by agents to complete complex user requests.
Enterprise Security and Data Guardrail Metrics
Prompt Injection Prevention Rate: Number of malicious prompt overrides caught by the input regex guard (such as "ignore previous instructions").
Output PII and Credential Redactions: Count of sensitive items redacted from AI answers before reaching users (Emails, Phone numbers, Credit Cards, SSNs, API Keys, JWT Tokens, Connection Strings).
Content Safety Policy Violations: Number of prompts or responses blocked due to severity score 4 or higher in Azure AI Content Safety across Hate, SelfHarm, Sexual, or Violence categories.
Role-Based Governance and Quota Metrics
Daily Token Budget Exhaustion Count: Number of requests blocked when users hit their daily role quota (Engineer: 200,000 tokens/day; Tester: 500,000 tokens/day; Admin: 500,000 tokens/day; Default: 50,000 tokens/day).
Rate Limit Throttling Count (HTTP 429): Requests blocked due to exceeding Requests Per Minute limits (Engineer: 120 RPM; Tester: 300 RPM; Admin: 600 RPM; Default: 60 RPM).
Real-Time Quota Tracking: Real-time user token usage monitored via the GET /usage API.
HNSW Vector Semantic Cache Metrics
Semantic Cache Hit Ratio: Percentage of user queries answered directly from the RediSearch HNSW ANN vector index in Azure Cache for Redis Enterprise.
Cosine Similarity Score Distribution: Average similarity scores of query vectors evaluated against the 0.80 threshold.
Response Latency Reduction: Speed improvement of 5 millisecond Cache Hits versus 2.5 second LLM Cache Misses.
Azure OpenAI Cost Savings: Total tokens saved by serving cached answers.
System Health, Pipeline, and Telemetry Metrics
End-to-End Response Latency: Total request duration and Time to First Token (TTFT) streaming speed.
Document RAG Indexing Volume: Azure Document Intelligence OCR extraction speed and document chunk storage in parent_docs_store.
Attachment Upload Validation: SAS token generation volume and file upload validation (10 MB limit and PDF/image/text file types).
Multi-Service Health Status: System status (healthy versus degraded) for Azure Cosmos DB and Redis connection pools returned by GET /health.
Telemetry forwarding: Trace logs and error telemetry forwarded to Azure Application Insights and Langsmith.


# RAG evaluation metrics

## Context Precision

Context Precision measures how many of the retrieved documents are actually relevant to the user’s query.
It helps identify whether the system is returning useful information or unnecessary noise.
Higher context precision improves answer quality by ensuring the model focuses only on relevant context.

## Context Recall

Context Recall measures whether the system retrieves all the relevant information needed to answer a query.
It helps identify if important context is missing from the retrieved results.
Higher context recall ensures the model has enough information to generate complete and accurate answers.

## MRR (Mean Reciprocal Rank)

MRR measures how early the first relevant document appears in the ranked retrieval results.
It gives higher scores when the correct document is ranked at the top.
Higher MRR means users (and the LLM) can find useful information quickly.

## Hit rate@K

Hit Rate@K measures whether at least one relevant document appears in the top K retrieved results.
It checks if the system is able to “hit” the correct context within the first K results.
A higher Hit Rate@K means the retrieval system is more likely to provide useful information for answer generation.

## Context Relevancy
Measures how relevant the retrieved chunks are to the query.
Evaluates retrieval quality.

## Faithfulness (Groundedness)

Measures whether all claims in the generated answer are supported by the retrieved context.
Helps detect hallucinations.
High faithfulness means the model is not inventing information.

## Answer Relevancy
Checks if the generated answer directly addresses the user’s question.
Ensures the response stays on-topic.
High relevance means the answer matches the user’s intent. 

## Answer Correctness
Measures how close the generated answer is to the ground-truth answer.
Often evaluated using semantic similarity rather than exact match.
High correctness means the answer is factually accurate.

## Noise Robustness
Evaluates how well the model handles irrelevant or noisy retrieved context.
A robust system still produces correct answers despite distractions.
High robustness means better stability in real-world scenarios.




# Latency Optimization

## Azure Latency 

We reduced latency by initializing the Azure AI Content Safety and PII Detection clients once during FastAPI startup using the lifespan event, so the same connections are reused for all requests. We also execute the Content Safety and PII Detection API calls in parallel using asyncio.gather() instead of sequentially. This reduces overall response time and improves application performance.

## LLM Response Cache 

To reduce LLM latency and cost, we implement caching using Azure Cache for Redis. First, we check an exact-match cache for identical user queries. If a cached response exists, we return it immediately without calling the LLM. For semantically similar queries, we use embeddings to compare the new query with previously cached queries. If the similarity score is above a threshold, such as 0.95, we return the cached response instead of generating a new one. This significantly reduces response time, lowers Azure OpenAI token usage, and improves system throughput


## Caching History: 

Fetching chat history from Cosmos DB on every message slows things down. We could cache the recent session history in Azure Redis.

## LLM Latency 

To optimize it, we stream responses using Server-Sent Events so users receive tokens immediately, improving the Time to First Token. 
We use model routing by sending simple requests to smaller, faster models and complex requests to larger models. 

## Retrieval Latency  

We optimize Azure AI Search by applying metadata filters before vector search. By narrowing the search to relevant documents—for example, by department or document type—we reduce the search space, which improves retrieval speed and the relevance of the results

We optimize context size by retrieving multiple documents from Azure AI Search and using Azure Semantic Ranker to select only the most relevant chunks. By sending only the top 3–5 chunks to the LLM, we reduce prompt size, improve TTFT, lower token costs, and maintain answer quality

## Latency reduce

Here are the most effective strategies to reduce latency and speed up response times in your Enterprise Multi-Agent AI Assistant project:

1. Response Streaming (Time To First Token)
Instead of waiting 3 to 5 seconds for the LLM to generate the entire response, use Server-Sent Events (SSE) or WebSockets to stream tokens to the user interface one by one. This drops the perceived latency (Time to First Token) to under 400 milliseconds, keeping the user engaged while the rest of the answer generates.

2. Vector Semantic Caching (Instant Responses)
As implemented with Azure Cache for Redis Enterprise, caching AI responses using HNSW Vector Search allows repeated or semantically similar questions (Cosine Similarity >= 0.80) to be answered instantly in under 5 milliseconds, completely bypassing the LLM generation time.

3. Parallel Tool Execution (Async Processing)
When your Multi-Agent system needs to invoke multiple tools (for example, fetching data from a Database MCP and running a Tavily Web Search), use Python's asyncio to execute these tool calls concurrently rather than sequentially. 

4. Model Routing (Small vs Large Models)
Not every task requires the heaviest, slowest model like GPT-4o. Use a fast, lightweight model (like gpt-3.5-turbo, gpt-4o-mini, or local Ollama models) for basic routing, intent classification, and simple tool calling. Only route the query to the heavy, high-latency model when deep reasoning or complex multi-step synthesis is required.

5. Context Window Optimization and Reranking
The more text you send to the LLM, the longer it takes to process. Instead of blindly passing large document chunks, use a Semantic Reranker to filter the context down to only the top 3 most highly relevant paragraphs before sending the prompt to the LLM. Shorter input prompts process significantly faster.

6. Persistent Connection Pooling
Ensure your FastAPI backend uses persistent, kept-alive HTTP connection pools (like httpx.AsyncClient) when communicating with Azure OpenAI, Cosmos DB, and external MCP servers. This prevents the system from wasting 50 to 100 milliseconds negotiating new TLS handshakes on every single API request.

# Cache 

# Rate Limit 
Rate per request (Requests Per Minute / RPM) works by continuously tracking how many API calls a specific user makes within a rolling 60-second window.

Each user is assigned a maximum allowed number of requests per minute based on their role:
- Engineer role: maximum 120 requests per minute
- Tester role: maximum 300 requests per minute
- Admin role: maximum 600 requests per minute
- Default role: maximum 60 requests per minute

Every time a user sends an API request:
1. FastAPI identifies the user and their assigned role tier.
2. Redis looks up the timestamps of all requests made by this user in the last 60 seconds.
3. Any timestamps older than 60 seconds are automatically removed.
4. The timestamp of the new incoming request is recorded into Redis.
5. Redis counts the total active requests remaining in the 60-second window.
6. If the total request count exceeds the limit allowed for the user's role, the API immediately blocks the request and returns an HTTP 429 Too Many Requests error message before any LLM generation or database processing occurs.

This sliding window method continuously measures the exact previous 60 seconds, preventing users from bypassing rate limits with sudden burst traffic.

```python
async def enforce_rate_limit(current_user: User = Depends(validate_user)) -> None:
    # Get user role limit (e.g. 120 for Engineer)
    max_rpm = ROLE_LIMITS.get(current_user.role, ROLE_LIMITS["Default"])["max_rpm"]
    
    # Key in Redis for this user
    key = f"rate_limit:{current_user.id}"
    now = time.time()

    # Atomically clean old entries, record current call, and get current 60s total count
    async with redis.pipeline(transaction=True) as pipe:
        pipe.zremrangebyscore(key, 0, now - 60)  # Remove calls older than 60 seconds
        pipe.zadd(key, {str(now): now})          # Record this current call timestamp
        pipe.zcard(key)                         # Get total count in last 60 seconds
        pipe.expire(key, 60)                    # Auto-expire key after 60s
        results = await pipe.execute()

    total_requests_in_last_60s = results[2]

    # Block if over limit
    if total_requests_in_last_60s > max_rpm:
        raise HTTPException(
            status_code=429, 
            detail=f"Rate limit exceeded for role '{current_user.role}'."
        )
```

## Daily Token Budget
Daily Token Budgeting tracks the total cumulative number of tokens (input prompt tokens plus output AI response tokens) consumed by a specific user during a single calendar day.

Each user is assigned a maximum daily token limit based on their role:
- Engineer role: 200,000 tokens per day
- Tester role: 500,000 tokens per day
- Admin role: 500,000 tokens per day
- Default role: 50,000 tokens per day

How Daily Token Budgeting works step-by-step:

Before LLM Execution (Pre-Check Phase):
1. When a user submits a question, FastAPI checks Redis for the user's usage record for today's date.
2. If the user's recorded token usage has already reached or exceeded their assigned role limit, FastAPI immediately blocks the request and returns an HTTP 429 error stating that their daily token budget is exhausted.

After LLM Execution (Usage Recording Phase):
3. If the user has available budget, the AI model generates the response.
4. FastAPI calculates the exact number of total tokens consumed by counting both the user's question tokens and the AI answer tokens.
5. FastAPI automatically updates the user's daily token balance in Redis by adding the newly consumed tokens.
6. The token usage record in Redis automatically expires after 24 hours, ensuring that every user gets a fresh, full token allocation at midnight every single day.

## Sematic cache 

Semantic Caching uses AI vector embeddings and similarity algorithms in Azure Cache for Redis to deliver sub-second response times and eliminate unnecessary LLM costs for repeated or rephrased questions.

How Semantic Caching works step-by-step:

Step 1: Text Embedding Conversion
When a user submits a question, FastAPI converts the user's plain text prompt into a 1,536-dimensional numerical vector embedding using Azure OpenAI.

Step 2: Vector Search in Redis
FastAPI sends the question vector to Azure Cache for Redis Enterprise using an Approximate Nearest Neighbor (ANN) search algorithm called HNSW.
Redis compares the incoming question vector against all previously stored question vectors.

Step 3: Similarity Score Evaluation
- If the similarity score between the new question and a previously asked question is 0.80 or higher (80 percent or greater semantic match), it is a Cache Hit. The previously saved AI answer is returned to the user instantly in under 5 milliseconds without calling the LLM or spending tokens.
- If the highest similarity score is below 0.80, it is a Cache Miss.

Step 4: Fresh Generation and Cache Storage
On a Cache Miss, the system retrieves relevant document context and calls Azure OpenAI to generate a new answer.
FastAPI then saves both the new question vector and the generated AI answer into Redis with a 1-hour expiration time so future identical or rephrased questions will result in instant Cache Hits.

## Chat history cache

Chat History Caching stores recent message exchanges between a user and the AI assistant in fast Redis memory and Cosmos DB to provide multi-turn conversation context for the LLM.

How Chat History Caching works step-by-step:

Step 1: Conversation Session Tracking Every chat conversation is assigned a unique Session ID. All messages sent by the user and generated by the AI are tagged with this Session ID and a sequence number.

Step 2: Dual Storage Pattern

Permanent Database (Cosmos DB): Every single message, response, timestamp, and attachment is permanently saved in Cosmos DB partitioned by the User ID.
Fast Cache (Redis): The 10 most recent message exchanges for active sessions are cached in Redis in JSON format for instant retrieval.
Step 3: History Retrieval During Chat When a user asks a follow-up question in an ongoing chat:

FastAPI reads the last 10 messages for that session from Redis.
The recent conversation history is attached alongside the new question when calling the LLM.
This enables the LLM to understand follow-up questions, pronouns, and previous context (such as "Can you summarize the third point you mentioned above?").
Step 4: Automatic History Rotation Once the AI generates the new response, FastAPI appends the user question and AI answer to Cosmos DB and updates the Redis session cache, keeping only the 10 most recent messages active so prompt tokens stay low and efficient.

## Pdf Extraction 

This flow describes an Advanced Parent-Child Document Processing and Table Chunking Pipeline designed for Enterprise RAG systems.

Here is how each stage of the flow works:

Stage 1: Ingestion and Cloud Upload (Step 1)
The raw PDF file is uploaded directly to Azure Blob Storage, ensuring safe storage and generating a cloud URI for processing.

Stage 2: Layout Extraction and OCR (Step 2 and Step 3)
Azure AI Document Intelligence analyzes the PDF. It uses computer vision and optical character recognition (OCR) to understand the visual layout. It extracts text, section headings, multi-column paragraphs, and tables, converting everything into structured Markdown syntax. It then separates regular text paragraphs from structured tables.

Stage 3: Advanced Table Reconstruction and Chunking (Step 4 and Step 5)
- Table Merging: If a financial or technical table spans across multiple pages (e.g. Page 3 to Page 5), the system recognizes matching column headers, strips out duplicate repeated page headers, and stitches the table into a single continuous logical table.
- Table Chunking: To prevent large tables from overflowing LLM context windows, long tables are split into smaller chunks of 15 rows each. Crucially, the system prepends the main section heading and the column header row to every single chunk so each 15-row chunk remains fully self-describing.

Stage 4: Data Hygiene (Step 6)
Cleans up OCR artifacts, broken currency symbols, or formatting glitches to ensure high data quality before embedding.

Stage 5: Parent-Child Hierarchy (Step 7)
- Parent Document: Stores the full, complete table or large document section.
- Child Documents: Stores the smaller 15-row chunks and paragraph blocks.
- Parent ID Link: Every child chunk is tagged with a unique parent_id pointing to its parent document.
- Why this matters: Small child chunks are easy for vector search to find with pinpoint precision (high retrieval accuracy), while the LLM receives the complete parent document context to generate comprehensive answers (high generation quality).

Stage 6: AI Metadata Enrichment and HyDE (Step 8)
Before indexing, Azure OpenAI evaluates each chunk to generate three enrichment fields:
1. Summary: A short explanation of the chunk's content.
2. Keywords: Key domain concepts and acronyms.
3. HyDE (Hypothetical Document Embeddings / Sample Questions): 3 to 5 synthetic questions that this specific chunk can answer. Searching against questions rather than raw data dramatically improves search accuracy.

Stage 7: Vectorization and Hybrid Search Indexing (Step 9 and Step 10)
- Embeddings: Text content, summaries, and sample questions are converted into 3,072-dimensional vector embeddings using OpenAI text-embedding-3-large.
- Azure AI Search: The chunk text, vector embeddings, metadata, keywords, and parent_id are stored in Azure AI Search. This enables Hybrid Search (combining traditional BM25 keyword matching with dense vector similarity search).

Why Combine Azure AI Document Intelligence + PyMuPDF?
- Azure AI Document Intelligence serves as the AI Brain: It performs OCR on scanned pages, reads complex multi-column layouts, reconstructs tables, and outputs clean Markdown.
- PyMuPDF serves as the Fast File Engine: It directly accesses the raw PDF file stream to extract high-resolution embedded images (diagrams, logos, charts) in milliseconds without incurring AI API charges.

Combining both tools gives your RAG pipeline the highest quality text, table layout understanding, and image extraction at maximum speed and minimum cost.


## Azure Scaling

Here is the complete architectural scaling strategy for your enterprise RAG application across every tier:

1. Stateless Backend Scaling (FastAPI Tier)
- Horizontal Pod Autoscaling (KEDA): Deploy FastAPI on Azure Container Apps or Azure Kubernetes Service (AKS). Configure KEDA rules to automatically scale container instances from 2 up to 50+ pods based on CPU utilization (above 70 percent) or incoming request concurrency.
- Multi-Worker ASGI Execution: Run Uvicorn/Gunicorn with multiple worker processes per container instance to maximize CPU core utilization.
- Stateless Architecture: Because session state, authentication, rate limits, and token budgets live in Redis and Cosmos DB, any FastAPI instance can serve any user request seamlessly.

2. Caching and Rate Limiting Scaling (Azure Cache for Redis Enterprise)
- Cluster Sharding: Scale Azure Cache for Redis Enterprise horizontally by adding Redis shards to distribute key memory across multiple nodes.
- High-Performance HNSW Vector Cache: RediSearch HNSW vector indexes perform Approximate Nearest Neighbor (ANN) searches in logarithmic time. This allows Redis to serve up to 45 percent of incoming questions in under 5 milliseconds, offloading traffic from your database and Azure OpenAI.

3. Database Scaling (Azure Cosmos DB NoSQL)
- Autoscale Provisioned Throughput (RU/s): Enable Cosmos DB Autoscale, allowing database Request Units to automatically scale up (for example, from 400 RU/s to 10,000+ RU/s) during peak business hours and scale back down at night to minimize cost.
- Distributed Partitioning: Documents are partitioned by User ID and Session ID across physical partitions, ensuring sub-10 millisecond database reads and writes even as your database grows to millions of chat messages.

4. LLM Model Scaling (Azure OpenAI Tier)
- Multi-Region Load Balancing: Set up a round-robin load balancer across multiple Azure OpenAI regions (such as East US, Sweden Central, and West Europe). If one region hits its quota limit, traffic automatically fails over to another region.
- Provisioned Throughput Units (PTU): Switch high-traffic production models from Pay-As-You-Go to Azure OpenAI Provisioned Throughput Units (PTU). PTUs guarantee dedicated inference capacity and stable response times during heavy usage.

5. Document Processing Pipeline Scaling (Asynchronous Queues)
- Asynchronous Workers: Separate PDF document extraction from the main user-facing chat API. When a user uploads a document, FastAPI pushes an event to a Redis Queue or Azure Event Grid.
- Worker Pool Scaling: Dedicated background worker containers running Azure Document Intelligence and PyMuPDF process PDFs asynchronously without blocking live chat endpoints.

To handle 50,000+ requests, the application is designed as a stateless, horizontally scalable system. The FastAPI backend does not store any session data locally, so multiple instances can run behind an Azure Load Balancer. When traffic increases, Azure Container Apps, AKS, or Azure App Service automatically create more backend replicas, and incoming requests are distributed across them.

All shared state is stored in Azure Cache for Redis. Redis centrally manages rate limiting, token budgets, caching, and chat history, allowing every backend instance to access the same data without synchronization issues.

Persistent data is stored in Azure Cosmos DB and Azure Blob Storage. Cosmos DB automatically scales throughput and provides low-latency reads and writes, while Blob Storage efficiently stores large documents and files.

The application also includes protection mechanisms such as sliding-window rate limiting and token budgeting to prevent abuse and control LLM costs. If Redis becomes temporarily unavailable, the application follows a fail-open strategy to avoid downtime.

For background processing, Azure Functions can automatically scale to handle asynchronous workloads such as document ingestion and processing without impacting API performance.

## Azure Comos DB

We chose Azure Cosmos DB because our solution is completely Azure-based. It integrates seamlessly with Azure services, provides automatic scaling for unpredictable AI workloads, offers a flexible JSON document model for conversation history, delivers consistently low latency, and includes built-in high availability and global replication. PostgreSQL and MongoDB could also support the application, but Cosmos DB reduced operational complexity and aligned best with our cloud-native Azure architecture.

If an interviewer asks "Why use Azure Cosmos DB when we already have PostgreSQL and MongoDB?", here is how you can justify the architectural decision for an Enterprise AI project:

1. Multi-Model Support in a Single Service
"Cosmos DB isn't just one type of database. It natively supports both the MongoDB API and the PostgreSQL API. Instead of managing two completely separate database infrastructures, our team can use the exact same MongoDB and PostgreSQL drivers we already know, while Azure manages the underlying infrastructure in a unified way."

2. Built-in Vector Search for AI (RAG)
"For our AI Assistant, we need to store vector embeddings for Semantic Search and RAG. Both Cosmos DB for PostgreSQL (using pgvector) and Cosmos DB for MongoDB (vCore) have native, highly optimized Vector Search built directly into the database. This means we don't need to pay for and maintain a third standalone vector database (like Pinecone or Weaviate) just for our AI features—our transactional data and vector data live together."

3. Turnkey Global Distribution & Active-Active Replication
"Setting up active-active multi-region replication in open-source PostgreSQL or MongoDB is incredibly difficult and requires dedicated database administrators. Cosmos DB provides turnkey, one-click global distribution. We can replicate our AI database to any Azure region globally, giving users single-digit millisecond latency no matter where they are in the world."

4. Enterprise-Grade SLAs & Zero Maintenance
"Since we are building an Enterprise application, we need high reliability. Cosmos DB is a fully managed PaaS (Platform as a Service). It handles all patching, backups, and scaling automatically, and it is the only database that offers financially backed 99.999% SLAs for throughput, latency, consistency, and high availability."

5. Seamless Azure AI Ecosystem Integration
"Because we are using Azure OpenAI for our models, Cosmos DB integrates perfectly with the rest of the Azure ecosystem. We can use Azure Entra ID (Active Directory) for Role-Based Access Control (RBAC), and it connects natively to Azure AI Search, making our security and compliance much stronger."

Summary to memorize: "We chose Cosmos DB because it allows us to use our existing PostgreSQL and MongoDB code, while giving us native AI Vector Search, zero-maintenance global scaling, and 99.999% enterprise reliability that would be too hard to build ourselves from scratch."

## RBAC 
In my RAG application, RBAC is enforced during retrieval, not after generation. Every document is indexed with metadata such as allowed_roles, and when a user submits a query, their role is obtained from authentication and used as a metadata filter in Azure AI Search. Only authorized documents are retrieved and passed to the LLM, preventing unauthorized information from ever entering the prompt.

We use Azure DevOps for CI/CD. Developers create a feature branch, implement the feature, and raise a pull request. After code review, the changes are merged into the develop branch, which triggers the CI pipeline. The pipeline restores dependencies, builds the application, runs unit tests and security scans, builds a Docker image, pushes it to Azure Container Registry, and publishes the build artifact. The release pipeline then deploys the application to the DEV environment for QA and integration testing. After successful validation and approvals, the same tested artifact is deployed to the staging environment. Smoke tests are executed, and once approved, Azure performs a slot swap from staging to production, ensuring a zero-downtime deployment.

During development, we deploy the application on Azure App Service because it's simple to set up, supports CI/CD easily, and is sufficient for development and testing. For production, we deploy on Azure Container Apps because our application is containerized, requires auto-scaling, supports multiple microservices, and provides better scalability and cost efficiency for production workloads.

Blue-Green deployment is a zero-downtime deployment strategy where two identical environments are maintained. The Blue environment runs the current production version, while the Green environment hosts the new version. After testing the Green environment, traffic is switched to it. If any issues occur, traffic can be immediately routed back to the Blue environment, enabling fast rollback with minimal downtime.